In [1]:
pip install mlflow

  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached pandas-2.3.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_proto-1.39.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached sqlp

  You can safely remove it manually.


In [ ]:
import pandas as pd
import os
import zipfile
import numpy as np
import torch
from torch.utils.data import Dataset
import timm
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import mlflow

In [3]:
batch_size = 8

In [4]:
DATASET_PATH = "jaguar-re-id.zip"
EXTRACT_PATH = "data"

if not os.path.exists(EXTRACT_PATH):
    print("Разархивирование датасета...")
    with zipfile.ZipFile(DATASET_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print(f"Датасет разархивирован в {EXTRACT_PATH}")
else:
    print(f"Датасет уже находится в {EXTRACT_PATH}")

Датасет уже находится в data


In [5]:
path_train = 'data/train.csv'
path_test = 'data/test.csv'
train = pd.read_csv(path_train)
test = pd.read_csv(path_test)

In [6]:
encoder = LabelEncoder()

In [7]:
train['ground_truth'] = encoder.fit_transform(train['ground_truth'])

In [8]:
train.head()

,filename,ground_truth
0,train_0001.png,0
1,train_0002.png,0
2,train_0003.png,0
3,train_0004.png,1
4,train_0005.png,1


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    train['filename'], train['ground_truth'], test_size=0.15, random_state=12)

In [10]:
transform = T.Compose([T.Resize(size=(384, 384)),
                              T.ToTensor(), 
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

In [11]:
class CustomDataset(Dataset):
    def __init__(self, X=X_train, y=y_train, img_dir='data/train/train'):
        self.X = X
        self.y = y
        self.img = img_dir
        self.transform = T.Compose([T.Resize(size=(384, 384)),
                              T.ToTensor(), 
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])]) 
         
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        img_path = self.X.iloc[idx]
        img = Image.open(self.img + '/' + img_path).convert('RGB')
        return {'x': self.transform(img), 'y': self.y.iloc[idx]}

In [12]:
dataset_1 = CustomDataset()

In [ ]:
dataset_test = CustomDataset(X_test, y_test, 'data/test/test')

In [13]:
dataset_1.__len__()

1610

In [14]:
dataset_1.__getitem__(idx=1123)

{'x': tensor([[[ 0.2863,  0.3176,  0.3725,  ..., -0.1294, -0.1059, -0.0824],
          [ 0.0353,  0.1216,  0.2235,  ..., -0.2078, -0.1765, -0.0431],
          [-0.0980, -0.0824, -0.0353,  ..., -0.3255, -0.1765,  0.0039],
          ...,
          [ 0.1686,  0.0902,  0.0667,  ..., -0.2549, -0.2078, -0.2078],
          [ 0.1529,  0.1294,  0.1373,  ..., -0.2863, -0.1843, -0.1529],
          [ 0.1373,  0.1216,  0.1294,  ..., -0.2392, -0.1922, -0.1843]],
 
         [[ 0.2078,  0.2392,  0.2941,  ..., -0.2078, -0.1843, -0.1608],
          [-0.0667,  0.0118,  0.1137,  ..., -0.2863, -0.2549, -0.1216],
          [-0.2314, -0.2078, -0.1608,  ..., -0.4039, -0.2549, -0.0824],
          ...,
          [ 0.1373,  0.0588,  0.0353,  ..., -0.3412, -0.2941, -0.2941],
          [ 0.1137,  0.0902,  0.0980,  ..., -0.3725, -0.2706, -0.2392],
          [ 0.1059,  0.0824,  0.0902,  ..., -0.3255, -0.2784, -0.2706]],
 
         [[ 0.1373,  0.1843,  0.2627,  ..., -0.2235, -0.2000, -0.1765],
          [-0.1529, -0.

In [15]:
from transformers import DefaultDataCollator

In [16]:
data_collator = DefaultDataCollator()

In [17]:
train_dataloader = DataLoader(dataset_1, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

In [ ]:
test_dataloader = DataLoader(dataset_test, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

In [20]:
model = timm.create_model("hf-hub:BVRA/MegaDescriptor-L-384", pretrained=True, num_classes=31)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
from transformers import TrainingArguments

# Показываем потери при обучении для каждой эпохи
logging_steps = train.shape[0] // batch_size
model_name = 're_identification_v1'

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    logging_steps=logging_steps,
    num_train_epochs=5,
    fp16=torch.cuda.is_available(),
)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_1,
    compute_metrics=compute_metrics,
    train_dataset=dataset_1,
    eval_dataset=dataset_test,
)

In [ ]:
MLFLOW_EXPERIMENT_NAME = "re_identification_finetuning"
MLFLOW_TRACKING_URI = "mlruns"

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

In [ ]:
import datetime

In [ ]:
with mlflow.start_run(run_name=f"MegaDescriptor-L-384_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
    mlflow.log_params({
        "model_name": "MegaDescriptor-L-384",
        "learning_rate": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "num_epochs": training_args.num_train_epochs,
        "weight_decay": training_args.weight_decay
        })
    mlflow.log_dict({str(i): label for i, label in enumerate(label_encoder.classes_)},
        "label_mapping.json")
    train_result = trainer.train()
    mlflow.log_metrics({
    "train_loss": train_result.training_loss,
    "train_runtime": train_result.metrics.get('train_runtime', 0),
    "train_samples_per_second": train_result.metrics.get('train_samples_per_second', 0),
    })
    val_metrics = trainer.evaluate()
    mlflow.log_metrics({
        "val_loss": val_metrics['eval_loss'],
        "val_accuracy": val_metrics['eval_accuracy'],
        "val_f1": val_metrics['eval_f1'],
        "val_precision": val_metrics['eval_precision'],
        "val_recall": val_metrics['eval_recall'],
    })
    print(f"\n📊 Метрики валидации:")
    print(f"   Loss: {val_metrics['eval_loss']:.4f}")
    print(f"   Accuracy: {val_metrics['eval_accuracy']:.4f}")
    print(f"   F1: {val_metrics['eval_f1']:.4f}")
    print(f"   Precision: {val_metrics['eval_precision']:.4f}")
    print(f"   Recall: {val_metrics['eval_recall']:.4f}")

In [23]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [24]:
from transformers import get_scheduler

In [25]:
num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1010


In [26]:
import torch.nn as nn
criterion = nn.CrossEntropyLoss()

In [27]:
torch.cuda.empty_cache()

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.to('cuda')
for epoch in range(num_epochs):
    model.train()
    for batch in train_dataloader:
        data = batch
        X = data['x'].to('cuda')
        y = data['y'].to('cuda')
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update()
        print(loss.item())
        
    model.eval()
    for batch in test_dataloader:
        data = batch
        X = data['x'].to('cuda')
        y = data['y'].to('cuda')
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update()
        print(loss.item())
    
    torch.cuda.empty_cache()

In [ ]:
criterion(outputs, batch['y'])

tensor(3.7480, grad_fn=<NllLossBackward0>)